In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_context("notebook", font_scale=1.1)


# Data pre-processing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Feature engineering
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer

# modelling specifics
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Load the dataset
df_original = pd.read_csv('RFLFSODataFull.csv')
df = df_original.sample(frac=0.1, random_state=42)

# Data Preprocessing
def preprocess_data(df):
    SYNOP_mapping = {
        0: 'Clear',
        3: 'Dust Storm',
        4: 'Fog',
        5: 'Drizzle',
        6: 'Rain',
        7: 'Snow',
        8: 'Showers'
    }
    df['SYNOPCode'] = df['SYNOPCode'].map(SYNOP_mapping)
    df = pd.get_dummies(df, columns=['SYNOPCode'], drop_first=True)
    return df

df = preprocess_data(df)


# Feature Scaling
scaler = StandardScaler()

In [10]:
# Display initial data statistics and first few rows for exploration
df_stats = df.describe()
df_head = df.head()

df_stats, df_head

(           FSO_Att      RFL_Att  AbsoluteHumidity  AbsoluteHumidityMax  \
 count  9138.000000  9138.000000       9138.000000          9138.000000   
 mean      6.793461    11.617614          9.582060            10.065802   
 std       3.933853     3.456588          5.863303             6.171242   
 min       0.840225     1.361607          1.141556             1.238270   
 25%       3.501145    10.804116          4.971689             5.218967   
 50%       6.343527    11.868339          6.919733             7.258479   
 75%       8.673891    12.834316         14.001838            14.725286   
 max      31.646033    46.853960         24.307227            25.953720   
 
        AbsoluteHumidityMin     Distance     Frequency  Particulate  \
 count          9138.000000  9138.000000  9.138000e+03  9138.000000   
 mean              9.099081  3308.448141  7.846389e+10    27.430165   
 std               5.578477  1222.483034  5.000143e+09    73.717604   
 min               1.049744  2012.00545

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define the preprocessing steps based on the provided guidance

def preprocess_data(df, num_features, cat_features):
    # Convert NaN values to a new category for categorical variables 'SYNOP Code'
    df['SYNOPCode'] = df['SYNOPCode'].fillna('missing').astype(str)
    SYNOP_mapping = {
    0: 'Clear',
    3: 'Dust Storm',
    4: 'Fog',
    5: 'Drizzle',
    6: 'Rain',
    7: 'Snow',
    8: 'Showers'
}

    # Replace the numerical categories with strings
    df['SYNOPCode'] = df['SYNOPCode'].map(SYNOP_mapping).fillna(df['SYNOPCode'])

    # Define preprocessor
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    # Create and return the column transformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features),
            ('cat', cat_transformer, cat_features)
        ])

    return preprocessor

# Define preprocessing functions for fso and rfl
def preprocess_fso(df):
    # Apply general preprocessing
    preprocessor = preprocess_data(df, num_features, cat_features)
    
    # Additional preprocessing specific to fso can be added here
    
    return preprocessor

def preprocess_rfl(df):
    # Apply general preprocessing
    preprocessor = preprocess_data(df, num_features, cat_features)
    
    # Additional preprocessing specific to rfl can be added here
    
    return preprocessor

preprocess_fso, preprocess_rfl


In [3]:
# Importing necessary libraries for data processing, feature engineering, and model building
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import numpy as np

# Load the dataset


# Function to build and evaluate a Random Forest model
def build_and_evaluate_rf_model(df, target_column, scaler, preprocess_func=None):
    # Data Preprocessing
    if preprocess_func:
        df = preprocess_func(df)
        
    
    # Splitting the data
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Feature Scaling
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Model Building
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    
    # Prediction
    y_pred = model.predict(X_test)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred, multi_class='ovr', average='weighted')
    
    # Feature Importance
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    plt.figure()
    plt.title(f"Feature importances for {target_column}")
    plt.bar(range(X_train.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train.shape[1]), df.columns[indices], rotation=45)
    plt.show()
    
    return {
        'Model': model,
        'Metrics': {
            'Accuracy': accuracy,
            'F1 Score': f1,
            'Confusion Matrix': cm,
            'ROC AUC Score': roc_auc
        }
    }

# Placeholder functions for data preprocessing and feature engineering (to be filled based on existing models)
def preprocess_fso(df):
    # Placeholder
    return df

def preprocess_rfl(df):
    # Placeholder
    return df


    return df

# Initialize StandardScaler
scaler = StandardScaler()

# Build and evaluate models
fso_model_result = build_and_evaluate_rf_model(df, 'FSO_Att', scaler, preprocess_fso)
rfl_model_result = build_and_evaluate_rf_model(df, 'RFL_Att', scaler, preprocess_rfl)

fso_model_result['Metrics'], rfl_model_result['Metrics']


ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [ ]:
# Importing RandomForestRegressor for regression tasks
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Function to build and evaluate a Random Forest Regressor model
def build_and_evaluate_rf_regressor(df, target_column, scaler, preprocess_func=None, feature_eng_func=None):
    # Data Preprocessing
    if preprocess_func:
        df = preprocess_func(df)
        
    # Feature Engineering
    if feature_eng_func:
        df = feature_eng_func(df)
    
    # Splitting the data
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Feature Scaling
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Model Building
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    
    # Prediction
    y_pred = model.predict(X_test)
    
    # Metrics
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Feature Importance
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    plt.figure()
    plt.title(f"Feature importances for {target_column}")
    plt.bar(range(X_train.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train.shape[1]), X.columns[indices], rotation=45)
    plt.show()
    
    return {
        'Model': model,
        'Metrics': {
            'Mean Squared Error': mse,
            'Mean Absolute Error': mae,
            'R2 Score': r2
        }
    }

# Build and evaluate models again, this time using RandomForestRegressor
fso_model_result = build_and_evaluate_rf_regressor(df, 'FSO_Att', scaler, preprocess_fso, feature_engineer_fso)
rfl_model_result = build_and_evaluate_rf_regressor(df, 'RFL_Att', scaler, preprocess_rfl, feature_engineer_rfl)

fso_model_result['Metrics'], rfl_model_result['Metrics']
